In [ ]:
# --- Paths (repo-relative; this notebook runs from notebooks/) ---
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA      = PROJECT_ROOT / 'data'
RAW       = DATA / 'raw'          # external source data (read-only)
EXTERNAL  = DATA / 'external'     # frozen third-party / HPC-derived inputs (read-only)
ANNOTATED = DATA / 'annotated'    # tables derived by these notebooks
FIGURES   = PROJECT_ROOT / 'figures'


# Gene Age × IDR Analysis
Biological insights from IDRisoforms_df_geneage and TranscriptionIsoforms_df_geneage.

All figures saved to `figures/proteome_extended/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr, mannwhitneyu, gaussian_kde
from matplotlib.patches import Patch

In [ ]:
DATA_DIR = Path("../data/annotated")
outdir   = Path("../figures/proteome_extended")
outdir.mkdir(parents=True, exist_ok=True)

df_all = pd.read_csv(DATA_DIR / "IDRisoforms_df_geneage.csv")
df_tf  = pd.read_csv(DATA_DIR / "TranscriptionIsoforms_df_geneage.csv")

# One canonical row per gene, age known
can_all = df_all[df_all["is_canonical"] & df_all["age_bin"].notna()].copy()
tf_can  = df_tf[df_tf["is_canonical"]  & df_tf["age_bin"].notna()].copy()

print("All canonical genes:", len(can_all))
print("TF canonical genes: ", len(tf_can))

In [ ]:
AGE_ORDER  = ["<100", "100-500", "500-1000", ">1000"]
AGE_LABELS = ["<100 Ma\n(youngest)", "100-500 Ma", "500-1000 Ma", ">1000 Ma\n(oldest)"]
AGE_COLORS = ["#C6DBEF", "#6BAED6", "#2171B5", "#08306B"]   # sequential blue, light -> dark

TF_FILL    = "#DD8452"; TF_DARK    = "#A95A2C"
NONTF_FILL = "#4C72B0"; NONTF_DARK = "#2F4B7C"

plt.rcParams.update({
    "font.family":        "sans-serif",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.9,
    "xtick.major.size":   4,
    "ytick.major.size":   4,
})

def format_p(p):
    if p < 1e-300: return "p < 1e-300"
    if p < 0.001:  return f"p = {p:.2e}"
    return f"p = {p:.3f}"

def kde_width_at_y(vals, y, max_half_width):
    vals = np.asarray(vals.dropna())
    if len(vals) < 2 or np.std(vals) == 0:
        return max_half_width * 0.2
    kde = gaussian_kde(vals)
    y_grid = np.linspace(vals.min(), vals.max(), 300)
    return max_half_width * (kde([y])[0] / kde(y_grid).max())

def draw_violin(ax, vals, x, fill, dark, width=0.6):
    vals = vals.dropna()
    if len(vals) < 3:
        ax.scatter([x]*len(vals), vals, color=fill, s=20, alpha=0.6, zorder=3)
        return
    vp = ax.violinplot([vals], positions=[x], widths=width,
                       showmeans=False, showmedians=False, showextrema=False)
    body = vp["bodies"][0]
    body.set_facecolor(fill); body.set_edgecolor("black")
    body.set_linewidth(0.9);  body.set_alpha(0.80)
    hw = width / 2
    q1, med, q3 = np.percentile(vals, [25, 50, 75])
    for y in [q1, q3]:
        w = kde_width_at_y(vals, y, hw)
        ax.hlines(y, x-w, x+w, linewidth=1.1, color=dark, linestyles="dashed")
    w = kde_width_at_y(vals, med, hw)
    ax.hlines(med, x-w, x+w, linewidth=2.4, color=dark)

AGE_LEGEND = [Patch(facecolor=c, edgecolor="black", label=l)
              for c, l in zip(AGE_COLORS, ["<100 Ma","100-500 Ma","500-1000 Ma",">1000 Ma"])]

## Figure 1 — Gene age vs % IDR: opposing trends in TFs vs Non-TFs

In [ ]:
# ── Fig age1: % IDR vs gene age — TF vs Non-TF ───────────────────────────────
from matplotlib.lines import Line2D

# Sequential reds for age bins (light → dark)
RED_COLORS = ["#FCBBA1", "#FB6A4A", "#CB181D", "#67000D"]
RED_LEGEND = [Patch(facecolor=c, edgecolor="black", label=l)
              for c, l in zip(RED_COLORS, ["<100 Ma", "100–500 Ma", "500–1000 Ma", ">1000 Ma"])]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=False)

stats = {}
for ax, (grp, fill, dark) in zip(axes, [
        ("Non-TF", NONTF_FILL, NONTF_DARK),
        ("TF",     TF_FILL,    TF_DARK)]):

    gdf   = can_all[can_all["tf_group"] == grp]
    valid = gdf.dropna(subset=["gene_age", "pct_idr"])
    rho, p = spearmanr(valid["gene_age"], valid["pct_idr"])
    stats[grp] = (rho, p)

    for xi, b in enumerate(AGE_ORDER):
        draw_violin(ax, gdf[gdf["age_bin"] == b]["pct_idr"], xi, RED_COLORS[xi], RED_COLORS[xi])
        n = len(gdf[gdf["age_bin"] == b]["pct_idr"].dropna())
        ax.text(xi, -1.8, f"n={n:,}", ha="center", va="top", fontsize=8, color="gray")

    ax.set_xticks(range(4))
    ax.set_xticklabels(AGE_LABELS, fontsize=10)
    ax.set_ylabel("% IDR (canonical isoform)", fontsize=12)
    ax.set_title(grp, fontsize=14, color=dark, fontweight="bold")
    ax.set_ylim(-3, 105)

# ── Right-side legend: age colour key + labelled Spearman stats ───────────────
rho_nontf, p_nontf = stats["Non-TF"]
rho_tf,    p_tf    = stats["TF"]

legend_handles = (
    RED_LEGEND
    + [Patch(visible=False, label="")]          # blank spacer
    + [
        Line2D([], [], linestyle="none", marker="none", label="Spearman ρ / p"),
        Line2D([], [], color=NONTF_DARK, linewidth=2.5,
               label=f"Non-TF  ρ = {rho_nontf:.3f},  {format_p(p_nontf)}"),
        Line2D([], [], color=TF_DARK, linewidth=2.5,
               label=f"TF         ρ = {rho_tf:.3f},  {format_p(p_tf)}"),
    ]
)

axes[1].legend(
    handles=legend_handles,
    frameon=True, edgecolor="black", facecolor="white",
    loc="upper left", bbox_to_anchor=(1.03, 1.0),
    fontsize=9, handlelength=1.4,
    title="Age bin", title_fontsize=9,
)

fig.suptitle("Gene age vs. % IDR: opposing trends in TFs and Non-TFs",
             fontsize=14, y=1.01)
plt.tight_layout(rect=[0, 0, 0.83, 1])
fig.savefig(outdir / "fig_age1_pct_idr_vs_age.pdf", bbox_inches="tight")
plt.show()

## Figure 2 — Gene age vs isoform count (no significant relationship)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
valid = can_all.dropna(subset=["gene_age","n_isoforms"])
rho, p = spearmanr(valid["gene_age"], valid["n_isoforms"])

for xi, b in enumerate(AGE_ORDER):
    vals = can_all[can_all["age_bin"]==b]["n_isoforms"]
    draw_violin(ax, vals, xi, AGE_COLORS[xi], "#555555")
    ax.text(xi, -0.4, f"n={len(vals.dropna()):,}", ha="center", va="top", fontsize=8, color="gray")

ax.set_xticks(range(4)); ax.set_xticklabels(AGE_LABELS, fontsize=10)
ax.set_ylabel("Number of isoforms per gene", fontsize=12)
ax.set_title("Gene age vs. isoform count (all genes)", fontsize=13)
ax.set_ylim(-1, ax.get_ylim()[1])
ax.text(0.97, 0.97, f"Spearman rho = {rho:.3f}\n{format_p(p)} (n.s.)",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.3", alpha=1.0))
ax.legend(handles=AGE_LEGEND, frameon=True, edgecolor="black", facecolor="white",
          loc="upper left", bbox_to_anchor=(1.03, 1.0), fontsize=9)
plt.tight_layout(rect=[0, 0, 0.82, 1])
fig.savefig(outdir / "fig_age2_n_isoforms_vs_age.pdf", bbox_inches="tight")
plt.show()

## Figure 3 — IDR variation across isoforms vs. gene age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

for ax, (grp, fill, dark) in zip(axes, [
        ("Non-TF", NONTF_FILL, NONTF_DARK),
        ("TF",     TF_FILL,    TF_DARK)]):
    gdf   = can_all[can_all["tf_group"] == grp]
    valid = gdf.dropna(subset=["gene_age","pct_change_idr"])
    rho, p = spearmanr(valid["gene_age"], valid["pct_change_idr"])

    for xi, b in enumerate(AGE_ORDER):
        vals = gdf[gdf["age_bin"]==b]["pct_change_idr"]
        draw_violin(ax, vals, xi, AGE_COLORS[xi], AGE_COLORS[xi])
        ax.text(xi, -1.2, f"n={len(vals.dropna()):,}", ha="center", va="top", fontsize=8, color="gray")

    ax.set_xticks(range(4)); ax.set_xticklabels(AGE_LABELS, fontsize=10)
    ax.set_ylabel("IDR variation across isoforms (Delta% IDR)", fontsize=12)
    ax.set_title(grp, fontsize=14, color=dark, fontweight="bold")
    ax.set_ylim(-2, ax.get_ylim()[1])
    ax.text(0.97, 0.97, f"Spearman rho = {rho:.3f}\n{format_p(p)}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.3", alpha=1.0))

axes[1].legend(handles=AGE_LEGEND, frameon=True, edgecolor="black", facecolor="white",
               loc="upper left", bbox_to_anchor=(1.03, 1.0), fontsize=9)
fig.suptitle("IDR variation across isoforms vs. gene age:\nNon-TFs constrain, TFs stay flexible",
             fontsize=13, y=1.02)
plt.tight_layout(rect=[0, 0, 0.88, 1])
fig.savefig(outdir / "fig_age3_pct_change_idr_vs_age.pdf", bbox_inches="tight")
plt.show()

## Figure 4 — TF families ranked by evolutionary age

In [ ]:
family_stats = (tf_can.groupby("tf_family")
                .agg(median_age=("gene_age","median"), n=("gene_age","count"))
                .query("n >= 3")
                .sort_values("median_age"))

fig, ax = plt.subplots(figsize=(8, max(5, len(family_stats) * 0.32)))
norm   = plt.Normalize(family_stats["median_age"].min(), family_stats["median_age"].max())
colors = [plt.cm.Blues(norm(v)) for v in family_stats["median_age"]]

ax.barh(range(len(family_stats)), family_stats["median_age"],
        color=colors, edgecolor="black", linewidth=0.6, height=0.7)
ax.set_yticks(range(len(family_stats)))
ax.set_yticklabels(family_stats.index, fontsize=9)
ax.set_xlabel("Median gene age (Ma)", fontsize=12)
ax.set_title("TF families ranked by evolutionary age (families with >= 3 members)", fontsize=13)

for yi, (_, row) in enumerate(family_stats.iterrows()):
    ax.text(row["median_age"] + 15, yi, f"n={int(row['n'])}", va="center", fontsize=7.5, color="#444444")

sm = plt.cm.ScalarMappable(cmap=plt.cm.Blues, norm=norm); sm.set_array([])
plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.02).set_label("Median gene age (Ma)", fontsize=9)
plt.tight_layout()
fig.savefig(outdir / "fig_age4_tf_families_by_age.pdf", bbox_inches="tight")
plt.show()

## Figure 5 — IDR biophysical properties by gene age (heatmap)

In [ ]:
can_idr = can_all[can_all["n_idr_segments"] > 0]
props       = ["mean_FCR","mean_NCPR","mean_kappa","mean_fract_neg","mean_fract_pos","mean_fract_aro","mean_fract_pro"]
prop_labels = ["FCR\n(charged)","NCPR\n(net charge)","kappa\n(charge pattern)",
               "f(-)\n(neg)","f(+)\n(pos)","f(aro)\n(aromatic)","f(pro)\n(proline)"]

heat_data = np.array([[can_idr[can_idr["age_bin"]==b][p].median() for p in props] for b in AGE_ORDER])
heat_z    = (heat_data - heat_data.mean(0)) / (heat_data.std(0) + 1e-9)

fig, (ax_raw, ax_z) = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, data, title in [(ax_raw, heat_data, "Median values"), (ax_z, heat_z, "Z-score across age bins")]:
    vmax = np.abs(data).max() if ax is ax_z else None
    im = ax.imshow(data, aspect="auto",
                   cmap="RdBu_r" if ax is ax_z else "Blues",
                   vmin=-vmax if ax is ax_z else None, vmax=vmax if ax is ax_z else None)
    ax.set_xticks(range(len(props))); ax.set_xticklabels(prop_labels, fontsize=9)
    ax.set_yticks(range(4));          ax.set_yticklabels(AGE_LABELS, fontsize=9)
    ax.set_title(title, fontsize=12, pad=8)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
    fmt = ".3f" if ax is ax_raw else ".2f"
    for r in range(4):
        for c in range(len(props)):
            ax.text(c, r, format(data[r,c], fmt), ha="center", va="center", fontsize=8,
                    color="white" if abs(data[r,c]) > 0.6*np.abs(data).max() else "black")

fig.suptitle("IDR biophysical properties by gene age (canonical, IDR-containing proteins)",
             fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "fig_age5_idr_biophysics_heatmap.pdf", bbox_inches="tight")
plt.show()

## Figure 5b — Same heatmap, split by canonical vs. alternative isoform AND TF vs. Non-TF

Tests whether the age-bin biophysics pattern seen in Fig 5 (canonical, all genes pooled) is conserved when isoforms are split. All four panels share one color scale, so the same shade means the same z-score everywhere.

In [ ]:
# ── Fig age5b: IDR biophysics z-score heatmap — canonical vs alternative × TF vs Non-TF ──
def _age5b_group(tf_group, canonical):
    if canonical:
        sub = df_all[df_all["is_canonical"] & df_all["age_bin"].notna()]
    else:
        # alt isoforms carry no age_bin/gene_age of their own (gene-level property) —
        # join it from the canonical row of the same gene, same pattern as fig_age7
        canon_age = (df_all[df_all["is_canonical"]]
                     .dropna(subset=["age_bin"])
                     .set_index("base_accession")[["age_bin", "gene_age"]])
        sub = (df_all[~df_all["is_canonical"]]
               .drop(columns=["age_bin", "gene_age"], errors="ignore")
               .join(canon_age, on="base_accession"))
        sub = sub[sub["age_bin"].notna()]
    sub = sub[(sub["tf_group"] == tf_group) & (sub["n_idr_segments"] > 0)]
    heat = np.array([[sub[sub["age_bin"] == b][p].median() for p in props] for b in AGE_ORDER])
    z    = (heat - heat.mean(0)) / (heat.std(0) + 1e-9)
    n_by_bin = {b: len(sub[sub["age_bin"] == b]) for b in AGE_ORDER}
    return heat, z, n_by_bin

panel_order = [("TF", True), ("TF", False), ("Non-TF", True), ("Non-TF", False)]
titles = {("TF", True): "TF — canonical", ("TF", False): "TF — alternative",
          ("Non-TF", True): "Non-TF — canonical", ("Non-TF", False): "Non-TF — alternative"}
results = {g: _age5b_group(*g) for g in panel_order}

print("Sample sizes per panel (isoforms per age bin):")
for g in panel_order:
    print(f"  {titles[g]:26s} {results[g][2]}")

# Shared symmetric color scale across all 4 z-score panels so the same shade means
# the same thing everywhere (otherwise each panel auto-scales to its own max and
# "dark red" in one panel could mean a totally different z-score in another).
all_z = np.concatenate([results[g][1].ravel() for g in panel_order])
zmax  = np.nanmax(np.abs(all_z))

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
for ax, key in zip(axes.flat, panel_order):
    heat, z, n_by_bin = results[key]
    im = ax.imshow(z, aspect="auto", cmap="RdBu_r", vmin=-zmax, vmax=zmax)
    ax.set_xticks(range(len(props))); ax.set_xticklabels(prop_labels, fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(AGE_LABELS, fontsize=8)
    dark = TF_DARK if key[0] == "TF" else NONTF_DARK
    ax.set_title(titles[key], fontsize=11, color=dark, fontweight="bold")
    for r in range(4):
        for c in range(len(props)):
            ax.text(c, r, f"{z[r,c]:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(z[r,c]) > 0.6*zmax else "black")

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.03)
cbar.set_label("Z-score across age bins\n(SDs from that panel's own mean)", fontsize=9)
fig.suptitle("IDR biophysical properties by gene age — z-score heatmap\n"
             "Canonical vs. alternative isoforms, TF vs. Non-TF (shared color scale across panels)",
             fontsize=13, y=1.02)
fig.savefig(outdir / "fig_age5b_idr_biophysics_heatmap_canon_alt_tf_nontf.pdf", bbox_inches="tight")
plt.show()

## Figure 5c — Same layout as 5b, raw median values instead of z-score

In [ ]:
# Fig age5c: IDR biophysics MEDIAN heatmap -- canonical vs alternative x TF vs Non-TF
# Reuses _age5b_group() and results/panel_order/titles from the Fig 5b cell above.
# heat_data (index 0 of the returned tuple) holds raw per-property medians.

all_raw = np.concatenate([results[g][0].ravel() for g in panel_order])
raw_vmin, raw_vmax = np.nanmin(all_raw), np.nanmax(all_raw)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
for ax, key in zip(axes.flat, panel_order):
    heat, z, n_by_bin = results[key]
    im = ax.imshow(heat, aspect="auto", cmap="Blues", vmin=raw_vmin, vmax=raw_vmax)
    ax.set_xticks(range(len(props))); ax.set_xticklabels(prop_labels, fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(AGE_LABELS, fontsize=8)
    dark = TF_DARK if key[0] == "TF" else NONTF_DARK
    ax.set_title(titles[key], fontsize=11, color=dark, fontweight="bold")
    for r in range(4):
        for c in range(len(props)):
            ax.text(c, r, f"{heat[r,c]:.3f}", ha="center", va="center", fontsize=7.5,
                    color="white" if heat[r,c] > 0.6*raw_vmax else "black")

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.03)
cbar.set_label("Median value (raw units)", fontsize=9)
fig.suptitle("IDR biophysical properties by gene age - raw median values\nCanonical vs. alternative isoforms, TF vs. Non-TF (shared color scale across panels)",
             fontsize=13, y=1.02)
fig.savefig(outdir / "fig_age5c_idr_biophysics_median_heatmap_canon_alt_tf_nontf.pdf", bbox_inches="tight")
plt.show()

## Figure 6 — Maximum IDR length vs gene age: TFs accumulate longer IDRs

In [ ]:
# ── Fig age6: max IDR length vs gene age — all / canonical / alternative ──────
panels = [
    ("All isoforms",     df_all),
    ("Canonical only",   df_all[df_all["is_canonical"]]),
    ("Alternative only", df_all[~df_all["is_canonical"]]),
]

def _age6_panel(ax, source_df, title):
    idr_df = source_df[source_df["age_bin"].notna() & (source_df["n_idr_segments"] > 0)]
    for grp, fill, dark in [("Non-TF", NONTF_FILL, NONTF_DARK),
                             ("TF",     TF_FILL,    TF_DARK)]:
        gdf     = idr_df[idr_df["tf_group"] == grp]
        medians = [gdf[gdf["age_bin"] == b]["max_idr_len"].median()       for b in AGE_ORDER]
        q1s     = [gdf[gdf["age_bin"] == b]["max_idr_len"].quantile(0.25) for b in AGE_ORDER]
        q3s     = [gdf[gdf["age_bin"] == b]["max_idr_len"].quantile(0.75) for b in AGE_ORDER]
        ns      = [len(gdf[gdf["age_bin"] == b]["max_idr_len"].dropna())  for b in AGE_ORDER]
        ax.plot(range(4), medians, color=fill, linewidth=2.5, marker="o",
                markersize=8, markeredgecolor=dark, markeredgewidth=1.2,
                label=f"{grp} (median)", zorder=3)
        ax.fill_between(range(4), q1s, q3s, color=fill, alpha=0.18, label=f"{grp} IQR")
        # TF n on first row below axis, Non-TF on second row — both colored
        y_offset = -60 if grp == "TF" else -76
        for xi, n in enumerate(ns):
            ax.annotate(f"n={n}", xy=(xi, 0), xycoords=("data", "axes fraction"),
                        xytext=(0, y_offset), textcoords="offset points",
                        ha="center", fontsize=7.5, color=fill, fontweight="bold",
                        annotation_clip=False)
    ax.set_xticks(range(4)); ax.set_xticklabels(AGE_LABELS, fontsize=9)
    ax.set_ylabel("Longest IDR segment (aa)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
for ax, (title, src) in zip(axes, panels):
    _age6_panel(ax, src, title)

handles, labels = axes[0].get_legend_handles_labels()
axes[2].legend(handles, labels, frameon=True, edgecolor="black", facecolor="white",
               fontsize=9, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle("Maximum IDR segment length vs. gene age\n"
             "TF vs Non-TF — all isoforms / canonical only / alternative only",
             fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0, 0.88, 1])
fig.subplots_adjust(bottom=0.25)
fig.savefig(outdir / "fig_age6_max_idr_len_vs_age.pdf", bbox_inches="tight")
plt.show()

## Figure 7 — IDR is near-obligatory for TFs across all evolutionary ages

In [ ]:
# ── Fig age7: % isoforms with ≥1 IDR — genes / canonical / alternative ───────
# Three panels share the same grouped-bar style and y-axis (0–108%)

def _idr_pct_bars(ax, source_df, title, obs_label):
    """Grouped bar chart: % rows in source_df with n_idr_segments > 0, by age bin & tf_group."""
    x = np.arange(4)
    w = 0.35
    src = source_df[source_df["age_bin"].notna()].copy()

    for offset, (grp, fill, dark) in enumerate([
            ("Non-TF", NONTF_FILL, NONTF_DARK),
            ("TF",     TF_FILL,    TF_DARK)]):
        gdf  = src[src["tf_group"] == grp]
        pcts = [(gdf[gdf["age_bin"] == b]["n_idr_segments"] > 0).mean() * 100
                for b in AGE_ORDER]
        ns   = [len(gdf[gdf["age_bin"] == b]) for b in AGE_ORDER]
        pos  = x + (offset - 0.5) * w

        ax.bar(pos, pcts, width=w, color=fill, edgecolor=dark,
               linewidth=0.9, alpha=0.85, label=grp)

        for xi, (pct, n) in enumerate(zip(pcts, ns)):
            # percentage label above bar
            ax.text(pos[xi], pct + 0.8, f"{pct:.0f}%",
                    ha="center", va="bottom", fontsize=8, color=dark, fontweight="bold")
            # n label below x-axis
            ax.text(pos[xi], -2.5, f"n={n:,}",
                    ha="center", va="top", fontsize=7, color="gray")

    ax.axhline(100, color="gray", linewidth=0.7, linestyle=":", zorder=0)
    ax.set_xticks(x)
    ax.set_xticklabels(AGE_LABELS, fontsize=9)
    ax.set_ylim(-5, 112)
    ax.set_ylabel(f"{obs_label} with ≥1 IDR segment (%)", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.tick_params(axis="both", labelsize=9)

# ── Data sources for each panel ───────────────────────────────────────────────
# Panel 1: canonical only (one per gene, mirrors original fig7)
panel1_df = df_all[df_all["is_canonical"]].copy()
# Panel 2: canonical isoforms with age info attached
panel2_df = df_all[df_all["is_canonical"] & df_all["age_bin"].notna()].copy()
# Panel 3: alternative isoforms — join gene age from canonical partner
canon_age = (df_all[df_all["is_canonical"]]
             .dropna(subset=["age_bin"])
             .set_index("base_accession")[["age_bin", "gene_age"]])
alt_df = (df_all[~df_all["is_canonical"]]
          .drop(columns=["age_bin", "gene_age"], errors="ignore")
          .join(canon_age, on="base_accession"))
panel3_df = alt_df[alt_df["age_bin"].notna()].copy()

# ── 3-panel figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

_idr_pct_bars(axes[0], panel1_df,  "Genes\n(canonical isoform)",     "Genes")
_idr_pct_bars(axes[1], panel2_df,  "Canonical isoforms",              "Canonical isoforms")
_idr_pct_bars(axes[2], panel3_df,  "Alternative isoforms",            "Alternative isoforms")

# shared legend on last panel
axes[2].legend(
    handles=[Patch(facecolor=NONTF_FILL, edgecolor=NONTF_DARK, label="Non-TF"),
             Patch(facecolor=TF_FILL,    edgecolor=TF_DARK,    label="TF")],
    frameon=True, edgecolor="black", facecolor="white",
    fontsize=10, loc="upper left", bbox_to_anchor=(1.02, 1.0),
)

fig.suptitle("% isoforms with ≥1 IDR segment across gene age — TF vs Non-TF\n"
             "Left: genes (canonical)  ·  Centre: canonical isoforms  ·  Right: alternative isoforms",
             fontsize=12, y=1.02)
plt.tight_layout(rect=[0, 0, 0.91, 1])
fig.savefig(outdir / "fig_age7_pct_genes_with_idr.pdf", bbox_inches="tight")
plt.show()

## Figure 8 — Distribution of IDR segment counts: TF vs Non-TF
Normalized to % within each group (TF n=1,888; Non-TF n=19,913 — 10x size difference makes raw counts misleading).

In [ ]:
# ── Fig age8: n_idr_segments histogram — all / canonical / alternative ─────────
CAP  = 7
bins = np.arange(-0.5, CAP + 1.5, 1)

panels = [
    ("All isoforms",     df_all),
    ("Canonical only",   df_all[df_all["is_canonical"]]),
    ("Alternative only", df_all[~df_all["is_canonical"]]),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
for ax, (panel_title, source_df) in zip(axes, panels):
    for grp, fill, dark in [("Non-TF", NONTF_FILL, NONTF_DARK),
                             ("TF",     TF_FILL,    TF_DARK)]:
        raw      = source_df[source_df["tf_group"] == grp]["n_idr_segments"].dropna()
        vals     = raw.clip(upper=CAP)
        n        = len(vals)
        med      = raw.median()
        pct_zero = (raw == 0).mean() * 100
        weights  = np.full(n, 100 / n)
        ax.hist(vals, bins=bins, weights=weights, histtype="step",
                linewidth=2.2, color=fill,
                label=f"{grp}  n={n:,}  median={med:.0f}  ({pct_zero:.0f}% zero IDR)")
        ax.axvline(med, color=fill, linewidth=1.4, linestyle="--", alpha=0.75)
    ax.set_title(panel_title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Number of IDR segments per isoform", fontsize=11)
    ax.set_ylabel("Isoforms within group (%)", fontsize=11)
    ax.set_xticks(range(CAP + 1))
    ax.set_xticklabels([str(i) if i < CAP else f"{CAP}+" for i in range(CAP + 1)], fontsize=9)
    ax.legend(frameon=True, edgecolor="black", facecolor="white", fontsize=8.5, loc="upper right")
    ax.set_xlim(-0.5, CAP + 0.5)

fig.suptitle("Number of IDR segments per isoform: TF vs Non-TF\n"
             "(y-axis = % within each group — all / canonical / alternative)",
             fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "fig_age8_n_idr_segments_histogram.pdf", bbox_inches="tight")
plt.show()

## Figure A — Direction of IDR change in alternative isoforms vs. canonical

For each alternative isoform, compute pct_idr(alt) − pct_idr(canonical). Positive = alt gains disorder, negative = alt loses disorder.

In [ ]:
# ── Fig A: direction of IDR change in alt isoforms ────────────────────────────
canonical_pct = (df_all[df_all["is_canonical"]]
                 .set_index("base_accession")["pct_idr"]
                 .rename("canonical_pct_idr"))
alt_df = (df_all[~df_all["is_canonical"]]
          .join(canonical_pct, on="base_accession"))
alt_df["idr_delta"] = alt_df["pct_idr"] - alt_df["canonical_pct_idr"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (grp, fill, dark) in zip(axes, [("Non-TF", NONTF_FILL, NONTF_DARK),
                                         ("TF",     TF_FILL,    TF_DARK)]):
    deltas = alt_df[alt_df["tf_group"] == grp]["idr_delta"].dropna()
    p1, p99 = np.percentile(deltas, 1), np.percentile(deltas, 99)
    med = deltas.median()
    pct_pos = (deltas > 0).mean() * 100
    pct_neg = (deltas < 0).mean() * 100
    weights = np.full(len(deltas), 100 / len(deltas))

    ax.axvspan(p1, 0,   alpha=0.07, color="#E05C5C", zorder=0)
    ax.axvspan(0,  p99, alpha=0.07, color=fill,      zorder=0)
    ax.hist(deltas.clip(p1, p99), bins=60, weights=weights,
            color=fill, edgecolor="none", alpha=0.82)
    ax.axvline(0,   color="#333333", linewidth=1.5, linestyle="--", zorder=3, label=r"$\Delta$ = 0")
    ax.axvline(med, color=dark,      linewidth=1.5, linestyle="--", zorder=3,
               label=f"median = {med:+.1f}%")
    ax.set_title(grp, fontsize=14, color=dark, fontweight="bold")
    ax.set_xlabel(r"Alt isoform % IDR $-$ Canonical % IDR", fontsize=12)
    ax.set_ylabel("Isoforms within group (%)", fontsize=12)
    ax.text(0.97, 0.97,
            f"Alt gains IDR (>0): {pct_pos:.1f}%\nAlt loses IDR (<0): {pct_neg:.1f}%\nn = {len(deltas):,}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.3", alpha=1.0))
    ax.legend(frameon=True, edgecolor="black", facecolor="white", fontsize=9)

fig.suptitle("Direction of IDR change in alternative isoforms vs. canonical", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "figA_idr_change_direction.pdf", bbox_inches="tight")
plt.show()

## Figure B — Do genes with more isoforms explore more IDR space?

Scatter of 
_isoforms vs pct_change_idr (one point per gene). Regression line + Spearman rho per group.

In [ ]:
# ── Fig B: n_isoforms vs pct_change_idr ──────────────────────────────────────
from scipy.stats import linregress

gene_df = df_all[df_all["is_canonical"]].dropna(subset=["n_isoforms","pct_change_idr"])

fig, ax = plt.subplots(figsize=(8, 5.5))
for grp, fill, dark in [("Non-TF", NONTF_FILL, NONTF_DARK), ("TF", TF_FILL, TF_DARK)]:
    gdf = gene_df[gene_df["tf_group"] == grp]
    ax.scatter(gdf["n_isoforms"], gdf["pct_change_idr"],
               color=fill, alpha=0.18, s=10, zorder=2, rasterized=True)
    slope, intercept, _, _, _ = linregress(gdf["n_isoforms"], gdf["pct_change_idr"])
    rho, rho_p = spearmanr(gdf["n_isoforms"], gdf["pct_change_idr"])
    xr = np.array([gdf["n_isoforms"].min(), gdf["n_isoforms"].max()])
    ax.plot(xr, slope*xr + intercept, color=dark, linewidth=2,
            label=f"{grp}  ρ={rho:.2f} {format_p(rho_p)}")

ax.set_xlabel("Number of isoforms per gene", fontsize=12)
ax.set_ylabel(r"IDR variation across isoforms ($\Delta$% IDR)", fontsize=12)
ax.set_title("More isoforms → more IDR variation?\n(one point per gene, canonical row)", fontsize=13)
ax.legend(frameon=True, edgecolor="black", facecolor="white", fontsize=10,
          loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout(rect=[0, 0, 0.82, 1])
fig.savefig(outdir / "figB_nisoforms_vs_pct_change_idr.pdf", bbox_inches="tight")
plt.show()

## Figure C — Where does the canonical isoform rank within its gene's IDR distribution?

For each multi-isoform gene, rank canonical by pct_idr among all isoforms. Stacked bar = % of genes in top 25%, middle 50%, bottom 25%.

In [ ]:
# ── Fig C: canonical IDR rank within gene ────────────────────────────────────
gene_ranks = []
for base_acc, grp_df in df_all.groupby("base_accession"):
    if len(grp_df) < 2: continue
    sorted_idr = grp_df["pct_idr"].sort_values().values
    canon_row  = grp_df[grp_df["is_canonical"]]
    if canon_row.empty: continue
    rank = np.searchsorted(sorted_idr, canon_row["pct_idr"].values[0]) / (len(sorted_idr) - 1)
    gene_ranks.append({"tf_group": grp_df["tf_group"].iloc[0], "canon_rank": rank})
rank_df = pd.DataFrame(gene_ranks)

def assign_tier(r):
    if r >= 0.75: return "Top 25%\n(most IDR)"
    if r <= 0.25: return "Bottom 25%\n(least IDR)"
    return "Middle 50%"

rank_df["tier"] = rank_df["canon_rank"].apply(assign_tier)
tier_order  = ["Top 25%\n(most IDR)", "Middle 50%", "Bottom 25%\n(least IDR)"]
tier_colors = ["#2171B5", "#6BAED6", "#C6DBEF"]

fig, ax = plt.subplots(figsize=(7, 5))
x_positions = {"Non-TF": 0, "TF": 1}
bottom = {"Non-TF": 0.0, "TF": 0.0}
legend_patches = []
for tier, color in zip(tier_order, tier_colors):
    for grp in ["Non-TF", "TF"]:
        sub = rank_df[rank_df["tf_group"] == grp]
        pct = (sub["tier"] == tier).mean() * 100
        ax.bar(x_positions[grp], pct, bottom=bottom[grp],
               color=color, edgecolor="black", linewidth=0.7, width=0.5)
        if pct > 4:
            ax.text(x_positions[grp], bottom[grp] + pct/2, f"{pct:.0f}%",
                    ha="center", va="center", fontsize=10, fontweight="bold", color="white")
        bottom[grp] += pct
    legend_patches.append(Patch(facecolor=color, edgecolor="black", label=tier.replace("\n"," ")))

ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-TF", "TF"], fontsize=12)
ax.set_ylabel("Genes (%)", fontsize=12); ax.set_ylim(0, 108)
ax.set_title("Where does the canonical isoform rank\nwithin its gene's IDR distribution?", fontsize=13)
ax.legend(handles=legend_patches, frameon=True, edgecolor="black", facecolor="white",
          fontsize=9, loc="upper left", bbox_to_anchor=(1.02, 1.0))
ns = rank_df.groupby("tf_group").size()
for grp, xi in x_positions.items():
    ax.text(xi, -3, f"n={ns.get(grp,0):,} genes", ha="center", fontsize=8, color="gray")
plt.tight_layout(rect=[0, 0, 0.82, 1])
fig.savefig(outdir / "figC_canonical_idr_rank.pdf", bbox_inches="tight")
plt.show()

## Figure D — IDR biophysical composition: TF vs Non-TF

Violins for FCR, κ, f(aro), f(neg), f(pos), NCPR. IDR-containing isoforms only. Mann-Whitney U p-value annotated.

In [ ]:
# ── Fig D: biophysical properties TF vs Non-TF ───────────────────────────────
idr_df = df_all[df_all["n_idr_segments"] > 0].copy()
props = [("mean_FCR",       "FCR\n(fraction charged residues)"),
         ("mean_kappa",     r"$\kappa$" + "\n(charge patterning)"),
         ("mean_fract_aro", "f(aromatic)\n(condensate proxy)"),
         ("mean_fract_neg", "f(negative)"),
         ("mean_fract_pos", "f(positive)"),
         ("mean_NCPR",      "NCPR\n(net charge per residue)")]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (prop, label) in zip(axes.flatten(), props):
    for xi, (grp, fill, dark) in enumerate([("Non-TF", NONTF_FILL, NONTF_DARK),
                                             ("TF",     TF_FILL,    TF_DARK)]):
        vals = idr_df[idr_df["tf_group"] == grp][prop].dropna()
        draw_violin(ax, vals, xi, fill, dark, width=0.55)
        ax.text(xi, ax.get_ylim()[0], f"med={vals.median():.3f}",
                ha="center", va="top", fontsize=7.5, color=dark)
    tf_v    = idr_df[idr_df["tf_group"]=="TF"][prop].dropna()
    nontf_v = idr_df[idr_df["tf_group"]=="Non-TF"][prop].dropna()
    _, p = mannwhitneyu(tf_v, nontf_v, alternative="two-sided")
    ax.text(0.97, 0.97, format_p(p), transform=ax.transAxes, ha="right", va="top",
            fontsize=8, bbox=dict(facecolor="white", edgecolor="black",
                                   boxstyle="round,pad=0.25", alpha=1.0))
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-TF", "TF"], fontsize=10)
    ax.set_title(label, fontsize=11)

fig.suptitle("IDR biophysical composition: TF vs Non-TF isoforms\n"
             "(IDR-containing isoforms only)", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "figD_biophysical_properties.pdf", bbox_inches="tight")
plt.show()

## Figure E — Condensate proxy scatter: aromatic fraction vs. charge fraction

Colored by % IDR. Density contours overlaid. TF isoforms may cluster toward high-FCR, high-aromatic space associated with transcriptional condensates (Boija et al.).

In [ ]:
# ── Fig E: condensate proxy scatter ──────────────────────────────────────────
idr_can = df_all[df_all["is_canonical"] & (df_all["n_idr_segments"] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True, sharex=True)
for ax, (grp, fill) in zip(axes, [("Non-TF", NONTF_FILL), ("TF", TF_FILL)]):
    gdf = idr_can[idr_can["tf_group"] == grp].dropna(subset=["mean_FCR","mean_fract_aro","pct_idr"])
    sc  = ax.scatter(gdf["mean_FCR"], gdf["mean_fract_aro"],
                     c=gdf["pct_idr"], cmap="YlOrRd",
                     s=14, alpha=0.55, vmin=0, vmax=100, rasterized=True)
    from scipy.stats import gaussian_kde as gkde
    try:
        xy  = np.vstack([gdf["mean_FCR"], gdf["mean_fract_aro"]])
        kde = gkde(xy)
        xi  = np.linspace(gdf["mean_FCR"].min(), gdf["mean_FCR"].max(), 60)
        yi  = np.linspace(gdf["mean_fract_aro"].min(), gdf["mean_fract_aro"].max(), 60)
        Xi, Yi = np.meshgrid(xi, yi)
        Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
        ax.contour(Xi, Yi, Zi, levels=4, colors=[fill], alpha=0.5, linewidths=0.8)
    except Exception:
        pass
    dark = TF_DARK if grp == "TF" else NONTF_DARK
    ax.set_title(grp, fontsize=14, color=dark, fontweight="bold")
    ax.set_xlabel("FCR (fraction charged residues)", fontsize=11)
    ax.set_ylabel("f(aromatic) — condensate proxy", fontsize=11)
    rho, p = spearmanr(gdf["mean_FCR"], gdf["mean_fract_aro"])
    ax.text(0.03, 0.97, f"ρ = {rho:.2f}  {format_p(p)}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.25"))

cbar = fig.colorbar(sc, ax=axes, fraction=0.015, pad=0.02)
cbar.set_label("% IDR", fontsize=10)
fig.suptitle("IDR condensate proxy: aromatic fraction vs. charge fraction\n"
             "(canonical isoforms with IDR, colored by % IDR)", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "figE_condensate_scatter.pdf", bbox_inches="tight")
plt.show()

## Figure F — IDR segment length: TF vs Non-TF

Longest and mean IDR segment length per isoform. Normalized histograms (% within group). IDR-containing isoforms only.

In [ ]:
# ── Fig F: IDR segment length distribution ───────────────────────────────────
idr_all = df_all[df_all["n_idr_segments"] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (col, title) in zip(axes, [("max_idr_len",  "Longest IDR segment per isoform"),
                                    ("mean_idr_len", "Mean IDR segment length per isoform")]):
    for grp, fill, dark in [("Non-TF", NONTF_FILL, NONTF_DARK),
                             ("TF",     TF_FILL,    TF_DARK)]:
        vals = idr_all[idr_all["tf_group"] == grp][col].dropna()
        p1, p99 = np.percentile(vals, 1), np.percentile(vals, 99)
        clipped = vals.clip(p1, p99)
        weights = np.full(len(clipped), 100 / len(clipped))
        med = vals.median()
        ax.hist(clipped, bins=50, weights=weights, histtype="step",
                linewidth=2.2, color=fill,
                label=f"{grp}  n={len(vals):,}  median={med:.0f} aa")
        ax.axvline(med, color=fill, linewidth=1.4, linestyle="--", alpha=0.75)

    tf_v    = idr_all[idr_all["tf_group"]=="TF"][col].dropna()
    nontf_v = idr_all[idr_all["tf_group"]=="Non-TF"][col].dropna()
    _, p = mannwhitneyu(tf_v, nontf_v, alternative="two-sided")
    ax.set_xlabel("Length (aa)", fontsize=12)
    ax.set_ylabel("Isoforms within group (%)", fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.text(0.97, 0.97, format_p(p), transform=ax.transAxes, ha="right", va="top",
            fontsize=9, bbox=dict(facecolor="white", edgecolor="black",
                                   boxstyle="round,pad=0.25", alpha=1.0))
    ax.legend(frameon=True, edgecolor="black", facecolor="white", fontsize=9)

fig.suptitle("IDR segment length distribution: TF vs Non-TF\n"
             "(IDR-containing isoforms only, p1–p99 shown)", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "figF_idr_segment_length.pdf", bbox_inches="tight")
plt.show()

## Figure G — TF family IDR profile heatmap

Z-scored median per column so columns with different scales are comparable. Right panel = n genes per family. Only families with ≥ 3 genes shown.

In [ ]:
# ── Fig G: TF family IDR heatmap ─────────────────────────────────────────────
tf_isoforms   = df_all[df_all["tf_group"] == "TF"].copy()
family_counts = tf_isoforms.groupby("tf_family")["base_accession"].nunique()
keep_families = family_counts[family_counts >= 3].index
tf_fam        = tf_isoforms[tf_isoforms["tf_family"].isin(keep_families)]

metrics = {
    "pct_idr":        "% IDR",
    "pct_change_idr": "Δ% IDR\n(isoform range)",
    "n_idr_segments": "# IDR\nsegments",
    "mean_FCR":       "FCR\n(charged)",
    "mean_fract_aro": "f(aro)\n(condensate)",
    "mean_kappa":     r"$\kappa$" + "\n(charge pattern)",
}
heat_data = (tf_fam.groupby("tf_family")[list(metrics.keys())]
             .median()
             .loc[family_counts[family_counts >= 3].sort_values(ascending=False).index])
heat_z = (heat_data - heat_data.mean()) / (heat_data.std() + 1e-9)

fig, (ax_z, ax_n) = plt.subplots(1, 2, figsize=(12, max(5, len(heat_z)*0.38)),
                                   gridspec_kw={"width_ratios": [5, 1]})
im = ax_z.imshow(heat_z.values, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax_z.set_xticks(range(len(metrics))); ax_z.set_xticklabels(list(metrics.values()), fontsize=9)
ax_z.set_yticks(range(len(heat_z)));  ax_z.set_yticklabels(heat_z.index, fontsize=9)
ax_z.set_title("TF family IDR profile (z-score per column)", fontsize=12)
for r in range(len(heat_z)):
    for c in range(len(metrics)):
        v = heat_z.values[r, c]
        ax_z.text(c, r, f"{v:.1f}", ha="center", va="center", fontsize=7.5,
                  color="white" if abs(v) > 1.5 else "black")
plt.colorbar(im, ax=ax_z, fraction=0.03, pad=0.02).set_label("Z-score", fontsize=9)

n_genes = family_counts[heat_z.index]
ax_n.barh(range(len(n_genes)), n_genes.values, color=TF_FILL, edgecolor=TF_DARK,
          linewidth=0.6, height=0.7)
ax_n.set_yticks(range(len(n_genes))); ax_n.set_yticklabels([])
ax_n.set_xlabel("n genes", fontsize=9); ax_n.set_title("n", fontsize=9)
for yi, n in enumerate(n_genes.values):
    ax_n.text(n + 0.3, yi, str(n), va="center", fontsize=7.5)
ax_n.spines["top"].set_visible(False); ax_n.spines["right"].set_visible(False)

fig.suptitle("IDR profile by TF family (families with ≥ 3 genes)", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(outdir / "figG_tf_family_idr_heatmap.pdf", bbox_inches="tight")
plt.show()

## Figure H — TF alternative isoforms: IDR properties by segment gain/loss

Split TF alt isoforms by whether they gained, lost, or kept the same number of IDR segments vs. the canonical. Compare % IDR, FCR, and aromatic fraction.

In [ ]:
# ── Fig H: TF alt isoforms by segment gain/loss ───────────────────────────────
from scipy.stats import kruskal

canon_seg = (df_all[df_all["is_canonical"]]
             .set_index("base_accession")["n_idr_segments"]
             .rename("canonical_n_seg"))
tf_alt = (df_tf[~df_tf["is_canonical"]]
          .join(canon_seg, on="base_accession")
          .dropna(subset=["canonical_n_seg"]))
tf_alt["seg_change"] = tf_alt["n_idr_segments"] - tf_alt["canonical_n_seg"]

def seg_label(x):
    if x < 0:  return "Lost segment(s)"
    if x > 0:  return "Gained segment(s)"
    return "Same count"

tf_alt["seg_group"] = tf_alt["seg_change"].apply(seg_label)
grp_order  = ["Lost segment(s)", "Same count", "Gained segment(s)"]
grp_colors = ["#E05C5C", "#AAAAAA", "#4C9A6A"]
grp_darks  = ["#A03030", "#666666", "#2E6645"]

fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))
for ax, (prop, title) in zip(axes, [
        ("pct_idr",        "% IDR"),
        ("mean_FCR",       "Charge (FCR)"),
        ("mean_fract_aro", "Aromatic fraction")]):
    for xi, (label, fill, dark) in enumerate(zip(grp_order, grp_colors, grp_darks)):
        sub = tf_alt[tf_alt["seg_group"] == label][prop].dropna()
        if len(sub) < 2: continue
        draw_violin(ax, sub, xi, fill, dark, width=0.55)
        ax.text(xi, ax.get_ylim()[0], f"n={len(sub):,}",
                ha="center", va="top", fontsize=8, color="gray")
    groups = [tf_alt[tf_alt["seg_group"]==g][prop].dropna().values for g in grp_order
              if len(tf_alt[tf_alt["seg_group"]==g][prop].dropna()) > 1]
    if len(groups) >= 2:
        _, p = kruskal(*groups)
        ax.text(0.97, 0.97, f"Kruskal-Wallis\n{format_p(p)}",
                transform=ax.transAxes, ha="right", va="top", fontsize=8,
                bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.25", alpha=1.0))
    ax.set_xticks(range(3))
    ax.set_xticklabels([g.replace(" ", "\n") for g in grp_order], fontsize=9)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(title, fontsize=11)

legend_els = [Patch(facecolor=c, edgecolor=d, label=l)
              for c, d, l in zip(grp_colors, grp_darks, grp_order)]
axes[2].legend(handles=legend_els, frameon=True, edgecolor="black", facecolor="white",
               fontsize=9, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle("TF alternative isoforms: IDR properties by segment gain/loss vs. canonical",
             fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0, 0.88, 1])
fig.savefig(outdir / "figH_tf_alt_seg_change.pdf", bbox_inches="tight")
plt.show()